# Face Recognition Based on OpenCV
This chapter introduces how to use OpenCV combined with ultraface-ncnn to compare a feature database and achieve face recognition functionality.

## Preparation
Since the product automatically runs the main program on startup, and the main program occupies the camera resources, this tutorial cannot be used in that state. You need to stop the main program or disable its automatic startup before restarting the robot.

It should be noted that because the robot's main program uses multithreading and is configured to start automatically via a service, the usual method of `sudo killall python` often does not work. Therefore, we will introduce a method to disable the main program's automatic startup.

If you have already disabled the automatic startup of the robot's main program, you do not need to perform the following "Stopping the Main Program" section.

### Stopping the Main Program
1. Click the "" icon next to the tab at the top of this page to open a new tab named Launcher.
2. Click Terminal under Other to open the terminal window.
3. In the terminal window, type `bash` and press Enter.
4. You can now use the Bash Shell to control the robot.
5. Enter the command: `systemctl --user stop ugv-app.service`.
6. On the terminal page, after the command has been executed, continue with the remaining part of this tutorial.

## Example

The following code block can be run directly:

1. Select the code block below.
2. Press Shift + Enter to run the code block.
3. Watch the real-time video window.
4. Press STOP to close the real-time video and release the camera resources.

### If you cannot see the real-time camera feed when running:

- Click on Kernel -> Shut down all kernels above.
- Close the current section tab and open it again.
- Click `STOP` to release the camera resources, then run the code block again.
- Reboot the device.

### Features of This Chapter

The face feature database file is located in the same path as this .ipynb file. You can change the faceCascade variable to modify what needs to be detected. You'll need to replace the current `haarcascade_frontalface_default.xml` file with other feature files.

When the code block runs successfully, you can position the robot's camera on a face, and the area containing the face will be automatically highlighted on the screen.

In [ ]:
import cv2  # Import the OpenCV library for image processing
import numpy as np  # Library for mathematical computations
from IPython.display import display, Image  # Used to display images in Jupyter Notebook
import ipywidgets as widgets  # Used to create interactive widgets, such as buttons
import threading  # Used to create new threads for asynchronous task execution
from UltraFaceNcnn import UltraFaceNcnn

face_detector = UltraFaceNcnn('./ultraface-ncnn/RFB-320.param','./ultraface-ncnn/RFB-320.bin', input_size=(320,240), threshold=0.7, nms_threshold=0.3)

# Create a 'Stop' button that the user can click to stop the video stream
# ================
stopButton = widgets.ToggleButton(
    value=False,
    description='Stop',
    disabled=False,
    button_style='danger', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Description',
    icon='square' # (FontAwesome names without the `fa-` prefix)
)


# Define the display function to process video frames and perform face detection
# ================
def view(button):

camera = cv2.VideoCapture(0)  # Create a camera instance
# Set resolution
camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

display_handle = display(None, display_id=True)  # Create a display handle to update the displayed image
i = 0

avg = None

while True:
    _, frame = camera.read()  # Capture a frame from the camera
    # frame = cv2.flip(frame, 1) # if your camera reverses your image

    # Use the cascade classifier for face detection
    faces = face_detector.detect(frame)

    if len(faces):
        for face in faces:  # Iterate over all detected faces
            cv2.rectangle(frame, (int(face.x1), int(face.y1)), (int(face.x2), int(face.y2)), (64, 128, 255), 1)

    _, frame = cv2.imencode('.jpeg', frame)  # Encode the frame in JPEG format
display_handle.update(Image(data=frame.tobytes()))
if stopButton.value == True:
    camera.release() # If so, close the camera
    display_handle.update(None)


# Display the "Stop" button and start the thread for the display function
# ================
display(stopButton)
thread = threading.Thread(target=view, args=(stopButton,))
thread.start()